# Model 2 service test

In [1]:
!pip -q install google-genai supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00


In [2]:
from google import genai
from google.genai import types
from google.colab import userdata

try:
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    print("Key found: yes")

    names = [m.name for m in client.models.list()
             if "generateContent" in (getattr(m, "supported_actions", None) or [])]
    flash = [n for n in names if "flash-lite" in n] or [n for n in names if "flash" in n]
    text_model = flash[0]
    print("Text model:", text_model)

    r = client.models.generate_content(model=text_model, contents="Reply with the single word OK")
    print("Text reply:", r.text.strip())

    e = client.models.embed_content(
        model="gemini-embedding-001", contents="test",
        config=types.EmbedContentConfig(output_dimensionality=768))
    print("Embedding length:", len(e.embeddings[0].values))
    print("GEMINI OK")
except Exception as err:
    print("GEMINI FAILED:", type(err).__name__, err)


Key found: yes
Text model: models/gemini-flash-lite-latest
Text reply: OK
Embedding length: 768
GEMINI OK


In [3]:
from supabase import create_client
from google.colab import userdata

try:
    sb = create_client(userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_SERVICE_KEY"))

    row = {"fact_id": "test-001", "livelihood": "TEST", "district": None, "scope": "test",
           "statement": "test row", "source": "test", "embedding": [0.1] * 768}
    sb.table("livelihood_facts").insert(row).execute()
    print("Insert OK")

    got = sb.table("livelihood_facts").select("fact_id,statement").eq("fact_id", "test-001").execute()
    print("Read back rows:", len(got.data))

    hits = sb.rpc("match_livelihood_facts", {"query_embedding": [0.1] * 768, "p_livelihood": "TEST",
                                             "p_district": "Any", "match_count": 1}).execute()
    print("Search OK, similarity:", round(hits.data[0]["similarity"], 3))

    sb.table("livelihood_facts").delete().eq("fact_id", "test-001").execute()
    left = sb.table("livelihood_facts").select("fact_id").eq("fact_id", "test-001").execute()
    print("Delete OK, rows left:", len(left.data))
    print("SUPABASE OK")
except Exception as err:
    print("SUPABASE FAILED:", type(err).__name__, err)


Insert OK
Read back rows: 1
Search OK, similarity: 1
Delete OK, rows left: 0
SUPABASE OK
